# 02 — Dataset Preparation

This notebook walks through:
1. Generating synthetic forged images
2. Inspecting class distribution and image quality
3. Splitting into train / val / test

In [2]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
import cv2

from src.data_generator import SyntheticForgeryGenerator
from src.utils import load_config

## 2.1 — Generate synthetic forgeries

In [3]:
# ---- point this at your downloaded real document images ----
REAL_DIR   = '../data/real'
OUTPUT_DIR = '../data/generated'
NUM_FORGED = 500   # adjust to your dataset size

gen = SyntheticForgeryGenerator(seed=42)

# Check real images are present
real_images = list(Path(REAL_DIR).glob('*.jpg')) + list(Path(REAL_DIR).glob('*.png'))
print(f'Real images found: {len(real_images)}')

if len(real_images) == 0:
    print('⚠  No real images found. Download a dataset first (see README).')
    print('   Example: Tobacco3482 or MIDV-500')
else:
    stats = gen.generate_dataset(
        real_dir=REAL_DIR,
        output_dir=OUTPUT_DIR,
        num_forged=NUM_FORGED,
    )
    print(f'Generated: {stats}')

Real images found: 0
⚠  No real images found. Download a dataset first (see README).
   Example: Tobacco3482 or MIDV-500


## 2.2 — Visualize forgery types

In [ ]:
# Create a quick demo with a dummy image
dummy = np.ones((300, 450, 3), dtype=np.uint8) * 240
cv2.putText(dummy, 'CONTRACT No. 12345', (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (30,30,30), 2)
cv2.putText(dummy, 'Signed: ___________', (30, 180), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (30,30,30), 1)
cv2.rectangle(dummy, (30, 240), (180, 290), (100,100,200), -1)  # fake stamp

forged_cm, meta_cm = gen.copy_move(dummy)
forged_sp, meta_sp = gen.splice(dummy, dummy[::-1, ::-1].copy())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes,
    [dummy, forged_cm, forged_sp],
    ['Original', 'Copy-Move Forgery', 'Splice Forgery']):
    ax.imshow(img); ax.set_title(title, fontsize=13); ax.axis('off')
plt.tight_layout()
plt.savefig('forgery_types.png', dpi=150, bbox_inches='tight')
plt.show()

print('Copy-move meta:', meta_cm)
print('Splice meta:   ', meta_sp)

## 2.3 — Class distribution & image statistics

In [ ]:
def count_images(directory: str) -> int:
    p = Path(directory)
    if not p.exists():
        return 0
    return len([f for f in p.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'}])

generated_real   = count_images(f'{OUTPUT_DIR}/real')
generated_forged = count_images(f'{OUTPUT_DIR}/forged')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Real', 'Forged'], [generated_real, generated_forged],
            color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Number of Images')
for i, v in enumerate([generated_real, generated_forged]):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
if generated_real + generated_forged > 0:
    axes[1].pie([generated_real, generated_forged],
                labels=['Real', 'Forged'],
                colors=['#2ecc71', '#e74c3c'],
                autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Class Balance')

plt.tight_layout()
plt.show()

balance_ratio = generated_real / max(generated_forged, 1)
print(f'Real:Forged ratio = {balance_ratio:.2f}')
if balance_ratio > 2 or balance_ratio < 0.5:
    print('⚠  Class imbalance detected — weighted loss is already handled in train_classifier.py')
else:
    print('✓  Classes are balanced')

## 2.4 — Run the train/val/test split

In [ ]:
import subprocess, sys

# This calls scripts/prepare_dataset.py
cmd = [
    sys.executable, '../scripts/prepare_dataset.py',
    '--source-dir', OUTPUT_DIR,
    '--output-dir', '../data',
    '--train-ratio', '0.70',
    '--val-ratio',   '0.15',
    '--seed',        '42',
]

if generated_real + generated_forged > 0:
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
else:
    print('⚠  No images to split. Generate data first (cell 2.1).')

## 2.5 — Preview training samples

In [ ]:
def show_grid(folder: str, n: int = 8, title: str = ''):
    paths = sorted(Path(folder).glob('*.jpg'))[:n] + sorted(Path(folder).glob('*.png'))[:n]
    paths = paths[:n]
    if not paths:
        print(f'No images in {folder}')
        return
    cols = 4
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).flatten()
    for ax, p in zip(axes, paths):
        ax.imshow(np.array(Image.open(p).resize((224, 224))))
        ax.axis('off')
    for ax in axes[len(paths):]:
        ax.axis('off')
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_grid('../data/train/real',   title='Training — Real')
show_grid('../data/train/forged', title='Training — Forged')